# Phase 2: Reproduce GNN-RAG

**Paper:** GNN-RAG: Graph Neural Retrieval for Large Language Modeling Reasoning (ACL 2025)
**Repo:** `colab/paper/GNN-RAG/`

## Muc tieu
1. Hieu kien truc GNN-RAG tu code thuc te
2. Reproduce results tren WebQSP
3. Extract GNN features cho Phase 4 (proposed method)
4. Phan tich diem manh/yeu -> inform proposed method

## Kien truc GNN-RAG (tu code thuc te)

Repo co 2 phan chinh:

### Phan 1: `gnn/` - Train GNN de score answer candidates
```
Question + Local KG Subgraph (per question)
   -> ReaRev GNN (multi-iteration reasoning):
      - num_iter iterations (default=3)
      - num_gnn GNN layers per iteration (default=3)
      - num_ins instruction vectors (query representations, default=2)
      - Instructions duoc UPDATE sau moi iteration (QueryReform)
   -> Output: probability distribution tren tat ca local entities
   -> Top candidates (entity, score) luu vao test.info
```

### Phan 2: `llm/` - RAG pipeline
```
GNN candidates (test.info)
   -> Shortest paths: question_entities -> candidate_answers (nx.all_shortest_paths)
   -> Verbalize paths: "entity1 -> relation -> entity2 -> ..."
   -> Token budget check (cat bot paths neu prompt qua dai)
   -> LLM generate answer (RoG/ChatGPT/LLaMA)
```

### Data format quan trong:
- Moi question co **local subgraph rieng** (khong phai 1 global KG)
- Data JSON: `{id, question, entities, answers, subgraph: {entities, tuples}}`
- GNN dung **global2local entity mapping** de xu ly batch
- Evaluation dung **entity-level F1** (khong phai token-level)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/GraphRAG_Research/utils')
from common import *

import json
import os
import pickle
import networkx as nx
import numpy as np
from collections import deque
from tqdm import tqdm

---
## 1. Setup GNN-RAG Repository

In [ ]:
# Copy repo tu Drive (hoac clone tu github)
GNN_RAG_DIR = '/content/GNN-RAG'

if not os.path.exists(GNN_RAG_DIR):
    # Option 1: Clone tu github
    !git clone https://github.com/cmavro/GNN-RAG.git {GNN_RAG_DIR}
    
    # Option 2: Copy tu Drive (neu da luu truoc do)
    # !cp -r /content/drive/MyDrive/GraphRAG_Research/repos/GNN-RAG {GNN_RAG_DIR}

print('=== Repo structure ===')
!ls {GNN_RAG_DIR}/
print('\n=== GNN module ===')
!ls {GNN_RAG_DIR}/gnn/
print('\n=== LLM module ===')
!ls {GNN_RAG_DIR}/llm/

In [ ]:
%%capture
# Install dependencies cho GNN part
!pip install -r {GNN_RAG_DIR}/gnn/requirements.txt

# Install dependencies cho LLM part
!pip install -r {GNN_RAG_DIR}/llm/requirements.txt

## 2. Download Data & Pre-trained Models

Tu README: data va pre-trained LM tai:
https://drive.google.com/drive/folders/1ifgVHQDnvFEunP9hmVYT07Y3rvcpIfQp

Pre-trained GNN models:
https://drive.google.com/file/d/1p7eLSsSKkZQxB32mT5lMsthVP6R_3x1j/view

In [ ]:
# === Download data tu Google Drive ===
# Ban can download thu cong va upload len Drive cua minh,
# hoac dung gdown:

!pip install -q gdown

GNN_DATA_DIR = os.path.join(GNN_RAG_DIR, 'gnn', 'data', 'webqsp')
os.makedirs(GNN_DATA_DIR, exist_ok=True)

# Download data folder
# gdown link: thay FOLDER_ID bang ID tu link Google Drive
# !gdown --folder https://drive.google.com/drive/folders/1ifgVHQDnvFEunP9hmVYT07Y3rvcpIfQp -O {GNN_RAG_DIR}/gnn/data/

# Download pre-trained GNN checkpoint
CKPT_DIR = os.path.join(GNN_RAG_DIR, 'gnn', 'checkpoint', 'pretrain')
os.makedirs(CKPT_DIR, exist_ok=True)
# !gdown 1p7eLSsSKkZQxB32mT5lMsthVP6R_3x1j -O {CKPT_DIR}/ReaRev_webqsp.ckpt

# Download entities_names.json (can cho llm part)
# !gdown ... -O {GNN_RAG_DIR}/llm/entities_names.json

print('=== Kiem tra data ===')
if os.path.exists(GNN_DATA_DIR):
    !ls -la {GNN_DATA_DIR}/
else:
    print(f'Data dir not found: {GNN_DATA_DIR}')
    print('Can download data tu Google Drive link o tren')

## 3. Hieu Data Format cua GNN-RAG

Moi question trong WebQSP co **local subgraph rieng** (khong phai 1 global KG).

In [ ]:
# === Doc 1 sample de hieu data format ===
DATA_FILE = os.path.join(GNN_DATA_DIR, 'test.json')

if os.path.exists(DATA_FILE):
    samples = []
    with open(DATA_FILE) as f:
        for i, line in enumerate(f):
            samples.append(json.loads(line))
            if i >= 2:
                break
    
    sample = samples[0]
    print('=== Sample keys ===')
    print(list(sample.keys()))
    
    print(f'\n=== Question ===')
    print(f'ID: {sample["id"]}')
    print(f'Question: {sample["question"]}')
    
    print(f'\n=== Entities (seed entities in question) ===')
    print(f'Num entities: {len(sample["entities"])}')
    for e in sample['entities'][:5]:
        print(f'  {e}')
    
    print(f'\n=== Answers ===')
    for a in sample['answers'][:5]:
        print(f'  {a}')
    
    print(f'\n=== Subgraph ===')
    subgraph = sample['subgraph']
    print(f'Entities in subgraph: {len(subgraph["entities"])}')
    print(f'Triples in subgraph: {len(subgraph["tuples"])}')
    
    print(f'\n=== Sample triples ===')
    for t in subgraph['tuples'][:5]:
        print(f'  {t}')
else:
    print(f'Data file not found: {DATA_FILE}')
    print('Download data truoc (cell o tren)')

## 4. Hieu ReaRev Architecture

ReaRev (Reasoning on Reversible graphs) la model GNN chinh trong GNN-RAG.

### So do:
```
Input: question tokens + local KG subgraph (adj matrix)
  |
  v
LM Encoder (SBERT/BERT/LSTM)
  -> question embedding
  -> num_ins instruction vectors (query representations)
  |
  v
For t = 1..num_iter:  <-- Multi-iteration reasoning
  |
  For j = 1..num_gnn:  <-- Multi-layer GNN per iteration
  |   ReasonGNNLayer:
  |     - Message passing tren local subgraph
  |     - Attention guided boi instruction vectors
  |     -> Updated entity distributions (curr_dist)
  |
  QueryReform:
  |   - Update instruction vectors dua tren graph context (global_rep)
  |   - Instruction[j] = reform(instruction[j], global_rep, query_entities)
  |
  v
Final: pred_dist = last curr_dist  <-- probability per local entity
Loss: KL-divergence(pred_dist, answer_dist)
```

### Key insight:
- **num_iter * num_gnn** = so hop thong tin co the di qua
  - Default: 3 iterations * 3 GNN layers = 9-hop reasoning
- **Instructions duoc update** -> query representation thay doi theo context da thay
- **Van la one-shot**: train xong -> inference 1 lan -> lay top candidates
- **Supervised**: train voi answer labels, khong co RL

In [ ]:
# === Hyperparameters cua ReaRev (tu parsing.py) ===

rearev_config = {
    'model_name': 'ReaRev',
    'entity_dim': 50,           # Entity embedding dim
    'num_epoch': 200,           # Training epochs
    'batch_size': 8,            # Batch size
    'eval_every': 2,            # Evaluate every N epochs
    'lm': 'sbert',              # Language model for encoding (sentence-transformers/all-MiniLM-L6-v2)
    'num_iter': 3,              # Reasoning iterations
    'num_ins': 2,               # Number of instruction vectors
    'num_gnn': 3,               # GNN layers per iteration
    'relation_word_emb': True,  # Use relation text embeddings
    'lr': 0.0005,               # Learning rate
    'gradient_clip': 1.0,       # Gradient clipping
    'loss_type': 'kl',          # KL-divergence loss
    'eps': 0.95,                # Threshold for F1 eval
    'alg': 'bfs',               # BFS-style message passing
}

print('=== ReaRev Config ===')
for k, v in rearev_config.items():
    print(f'  {k}: {v}')

total_hops = rearev_config['num_iter'] * rearev_config['num_gnn']
print(f'\nTotal reasoning hops: {rearev_config["num_iter"]} iter x {rearev_config["num_gnn"]} gnn = {total_hops}')

## 5. Train GNN (ReaRev) tren WebQSP

Command tu repo README:
```bash
python main.py ReaRev \
  --entity_dim 50 --num_epoch 200 --batch_size 8 --eval_every 2 \
  --data_folder data/webqsp/ --lm sbert \
  --num_iter 3 --num_ins 2 --num_gnn 3 \
  --relation_word_emb True --name webqsp
```

In [ ]:
%cd {GNN_RAG_DIR}/gnn

# === TRAIN ===
# Uncomment de train (mat ~2-4 gio tren A100)
# !python main.py ReaRev \
#     --entity_dim 50 --num_epoch 200 --batch_size 8 --eval_every 2 \
#     --data_folder data/webqsp/ --lm sbert \
#     --num_iter 3 --num_ins 2 --num_gnn 3 \
#     --relation_word_emb True --name webqsp

# === HOAC: Load pre-trained model ===
# Download checkpoint tu link trong README va dung --is_eval

print('Train hoac load pre-trained ReaRev model')
print('Neu train: checkpoint luu tai gnn/checkpoint/pretrain/')
print('Neu load: download tu Google Drive link trong README')

## 6. Evaluate GNN & Export test.info

Evaluate tao ra file `test.info` chua candidates + scores cho moi question.
File nay se duoc dung boi LLM pipeline.

In [ ]:
%cd {GNN_RAG_DIR}/gnn

# === EVALUATE ===
# Thay ReaRev_webqsp.ckpt bang ten checkpoint cua ban
# !python main.py ReaRev \
#     --entity_dim 50 --num_epoch 200 --batch_size 8 --eval_every 2 \
#     --data_folder data/webqsp/ --lm sbert \
#     --num_iter 3 --num_ins 2 --num_gnn 3 \
#     --relation_word_emb True \
#     --load_experiment ReaRev_webqsp.ckpt --is_eval --name webqsp

# Output: checkpoint/pretrain/{experiment_name}_test.info
# Copy sang llm/results/gnn/:
# !cp checkpoint/pretrain/*_test.info ../llm/results/gnn/RoG-webqsp/rearev-sbert/test.info

print('Evaluate GNN -> tao test.info file')
print('test.info chua: question, answers, candidates (entity, score), f1, hit')

## 6b. QUAN TRONG: Evaluate GNN tren TRAIN split

**De tranh data leakage cho Phase 4 (RL training):**
- Phase 4 can GNN features cho **training data** (khong phai test data)
- Phai chay GNN eval tren train split rieng -> tao `train.info`
- Dung train.info cho RL training, test.info chi cho evaluation

In [ ]:
%cd {GNN_RAG_DIR}/gnn

# === EVALUATE tren TRAIN split ===
# QUAN TRONG: can cho Phase 4 RL training (tranh data leakage)
#
# Cach 1: Sua code evaluate.py de chay tren train.json thay vi test.json
# Cach 2: Doi ten train.json -> test.json tam thoi, chay eval, doi lai
#
# !cp data/webqsp/test.json data/webqsp/test_backup.json
# !cp data/webqsp/train.json data/webqsp/test.json
# !python main.py ReaRev \
#     --entity_dim 50 --num_epoch 200 --batch_size 8 --eval_every 2 \
#     --data_folder data/webqsp/ --lm sbert \
#     --num_iter 3 --num_ins 2 --num_gnn 3 \
#     --relation_word_emb True \
#     --load_experiment ReaRev_webqsp.ckpt --is_eval --name webqsp_train_eval
# !cp data/webqsp/test_backup.json data/webqsp/test.json
#
# Output: checkpoint/pretrain/webqsp_train_eval_test.info
# Doi ten thanh train.info:
# !mv checkpoint/pretrain/webqsp_train_eval_test.info checkpoint/pretrain/train.info

print('Evaluate GNN tren TRAIN split -> train.info')
print('QUAN TRONG: Dung train.info cho Phase 4 RL training')
print('Chi dung test.info cho final evaluation')

## 7. Hieu test.info format (GNN output)

In [ ]:
# === Doc test.info de hieu GNN output format ===
# Moi dong la 1 JSON object cho 1 question

TEST_INFO_PATH = os.path.join(GNN_RAG_DIR, 'llm', 'results', 'gnn', 
                               'RoG-webqsp', 'rearev-sbert', 'test.info')

if os.path.exists(TEST_INFO_PATH):
    gnn_outputs = []
    with open(TEST_INFO_PATH) as f:
        for i, line in enumerate(f):
            gnn_outputs.append(json.loads(line))
            if i >= 2:
                break
    
    sample = gnn_outputs[0]
    print('=== GNN Output keys ===')
    print(list(sample.keys()))
    
    print(f'\nQuestion: {sample.get("question", "N/A")}')
    print(f'Answers: {sample.get("answers", [])}')
    print(f'F1: {sample.get("f1", "N/A")}')
    print(f'Hit: {sample.get("hit", "N/A")}')
    
    print(f'\n=== Candidates (entity, score) ===')
    cands = sample.get('cand', [])
    for c in cands[:10]:
        print(f'  {c[0]}: {c[1]:.4f}')
    print(f'Total candidates: {len(cands)}')
else:
    print(f'test.info not found at {TEST_INFO_PATH}')
    print('Can train/evaluate GNN truoc (step 5-6)')
    print('Hoac download pre-computed results tu repo')

## 8. LLM RAG Pipeline (Shortest Paths -> Verbalize -> LLM)

Day la phan **quan trong nhat** de hieu:
- Cach GNN-RAG chuyen GNN scores -> reasoning paths
- Cach verbalize paths thanh prompt cho LLM

### Pipeline (tu `llm/src/`):
```
1. Load GNN candidates (test.info) -> top entities voi scores
2. Map entity IDs -> entity names (entities_names.json)
3. Build graph tu question's subgraph (graph_utils.build_graph)
4. Tim shortest paths: question_entities -> candidate_answers
   (graph_utils.get_truth_paths dung nx.all_shortest_paths)
5. Verbalize: "entity1 -> relation -> entity2 -> relation2 -> entity3"
6. Check token budget, cat bot paths neu prompt qua dai
7. Prompt LLM: instruction + reasoning paths + question
```

In [ ]:
# === Re-implement core functions tu graph_utils.py ===
# De hieu ro va co the modify cho Phase 4

def build_graph_from_triples(triples):
    """Build NetworkX graph tu list of (head, relation, tail).
    Dung trong GNN-RAG: graph_utils.build_graph()
    """
    G = nx.Graph()  # Undirected (nhu GNN-RAG)
    for triplet in triples:
        h, r, t = triplet
        if isinstance(h, dict): h = h['text']
        if isinstance(r, dict): r = r['text']
        if isinstance(t, dict): t = t['text']
        G.add_edge(h, t, relation=r.strip())
    return G


def get_shortest_paths_with_relations(q_entities, a_entities, graph):
    """Tim shortest paths giua question entities va answer entities.
    Re-implement tu graph_utils.get_truth_paths()
    
    Returns: list of paths, moi path la list of (head, relation, tail)
    """
    paths = []
    for h in q_entities:
        if h not in graph:
            continue
        for t in a_entities:
            if t not in graph:
                continue
            try:
                for p in nx.all_shortest_paths(graph, h, t):
                    paths.append(p)
            except nx.NetworkXNoPath:
                pass
    
    # Add relations to paths
    result_paths = []
    for p in paths:
        triples = []
        for i in range(len(p) - 1):
            u, v = p[i], p[i+1]
            rel = graph[u][v].get('relation', 'related_to')
            triples.append((u, rel, v))
        result_paths.append(triples)
    return result_paths


def path_to_string(path):
    """Verbalize 1 path thanh string.
    Format: "entity1 -> relation -> entity2 -> relation -> entity3"
    """
    if not path:
        return ''
    parts = []
    for h, r, t in path:
        if not parts:
            parts.append(h)
        parts.append(r)
        parts.append(t)
    return ' -> '.join(parts)


def bfs_with_rule(graph, start_node, target_rule):
    """BFS tren graph theo rule (list of relations).
    Re-implement tu graph_utils.bfs_with_rule()
    
    Dung khi co predicted rules (tu RoG model): 
    follow relation sequence tu start_node.
    """
    result_paths = []
    queue = deque([(start_node, [])])
    while queue:
        current_node, current_path = queue.popleft()
        if len(current_path) == len(target_rule):
            result_paths.append(current_path)
        if len(current_path) < len(target_rule):
            if current_node not in graph:
                continue
            for neighbor in graph.neighbors(current_node):
                rel = graph[current_node][neighbor]['relation']
                if rel == target_rule[len(current_path)]:
                    queue.append((neighbor, 
                                  current_path + [(current_node, rel, neighbor)]))
    return result_paths


print('Core functions defined.')

In [ ]:
# === Demo: Full pipeline cho 1 question ===

DATA_FILE = os.path.join(GNN_DATA_DIR, 'test.json')

if os.path.exists(DATA_FILE):
    # Load 1 sample
    with open(DATA_FILE) as f:
        sample = json.loads(f.readline())
    
    print(f'Question: {sample["question"]}')
    
    # Step 1: Build graph tu local subgraph
    graph = build_graph_from_triples(sample['subgraph']['tuples'])
    print(f'\nLocal graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
    
    # Step 2: Question entities
    q_entities = []
    for e in sample['entities']:
        if isinstance(e, dict):
            q_entities.append(e['text'])
        else:
            q_entities.append(e)
    print(f'Question entities: {q_entities}')
    
    # Step 3: Answer entities
    a_entities = []
    for a in sample['answers']:
        if isinstance(a, dict):
            a_entities.append(a.get('text', a.get('kb_id', '')))
        else:
            a_entities.append(a)
    print(f'Answer entities: {a_entities}')
    
    # Step 4: Shortest paths
    paths = get_shortest_paths_with_relations(q_entities, a_entities, graph)
    print(f'\nFound {len(paths)} reasoning paths:')
    for i, p in enumerate(paths[:5]):
        print(f'  Path {i+1}: {path_to_string(p)}')
    
    # Step 5: Verbalize cho LLM
    all_paths_text = '\n'.join([path_to_string(p) for p in paths])
    prompt = f"""Based on the reasoning paths, please answer the given question.
Please keep the answer as simple as possible and return all the possible answers as a list.

Reasoning Paths:
{all_paths_text}

Question:
{sample['question']}"""
    
    print(f'\n=== Prompt for LLM ({len(prompt)} chars) ===')
    print(prompt[:500])
    if len(prompt) > 500:
        print('...')
else:
    print('Data not available. Download data truoc.')

## 9. Run Full GNN-RAG Pipeline (GNN + LLM)

In [ ]:
%cd {GNN_RAG_DIR}/llm

# === Run GNN-RAG voi RoG (LLM) ===
# Tu scripts/rag-reasoning.sh:

# GNN-RAG (chi GNN candidates, khong co RoG paths)
# !python src/qa_prediction/predict_answer.py \
#     --model_name RoG \
#     -d RoG-webqsp \
#     --prompt_path prompts/llama2_predict.txt \
#     --rule_path results/gen_rule_path/RoG-webqsp/RoG/test/predictions_3_False.jsonl \
#     --rule_path_g1 results/gnn/RoG-webqsp/rearev-sbert/test.info \
#     --rule_path_g2 None \
#     --model_path rmanluo/RoG \
#     --predict_path results/KGQA-GNN-RAG/rearev-sbert

# GNN-RAG-RA (GNN candidates + RoG predicted paths = Retrieval Augmentation)
# Them --add_rule de union paths

print('Run GNN-RAG LLM pipeline')
print('Output: results/KGQA-GNN-RAG/.../predictions.jsonl')

## 10. Entity-Level Evaluation (Dung cach cua paper)

**Quan trong:** GNN-RAG dung entity-level F1, KHONG PHAI token-level F1.

Tu `evaluate.py`:
- Predicted entities = candidates voi cumulative probability > eps (0.95)
- Precision = |predicted & gold| / |predicted|
- Recall = |predicted & gold| / |gold|
- F1 = harmonic mean
- Hits@1 = top-1 candidate co trong gold answers?

In [ ]:
# === Entity-level F1 (tu evaluate.py cua GNN-RAG) ===

def entity_f1_and_hits(gold_answers, candidate2prob, eps=0.95):
    """Entity-level F1 va Hits@1.
    
    Re-implement tu GNN-RAG evaluate.py:f1_and_hits()
    
    Args:
        gold_answers: set of gold answer entity IDs
        candidate2prob: list of (entity, probability) sorted by prob desc
        eps: threshold - lay candidates cho den khi cumulative prob > eps
    
    Returns:
        precision, recall, f1, hits@1, em
    """
    if len(gold_answers) == 0:
        if len(candidate2prob) == 0:
            return 1.0, 1.0, 1.0, 1.0, 1.0
        else:
            return 0.0, 1.0, 0.0, 1.0, 1.0
    
    # Sort by probability
    cand_list = sorted(candidate2prob, key=lambda x: -x[1])
    
    # Hits@1
    best_ans = cand_list[0][0] if cand_list else None
    hits = float(best_ans in gold_answers)
    
    # Retrieve candidates until cumulative prob > eps
    retrieved = []
    correct = 0
    tp_prob = 0.0
    for entity, prob in cand_list:
        retrieved.append(entity)
        tp_prob += prob
        if entity in gold_answers:
            correct += 1
        if tp_prob > eps:
            break
    
    # EM
    em = 1.0 if correct > 0 else 0.0
    
    # F1
    if len(retrieved) == 0:
        return 1.0, 0.0, 0.0, hits, hits
    
    precision = correct / len(retrieved)
    recall = correct / len(gold_answers)
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return precision, recall, f1, hits, em


# Test
p, r, f1, h, em = entity_f1_and_hits(
    gold_answers={'entity_A', 'entity_B'},
    candidate2prob=[('entity_A', 0.6), ('entity_C', 0.3), ('entity_B', 0.1)],
    eps=0.95
)
print(f'Test: P={p:.2f}, R={r:.2f}, F1={f1:.2f}, Hits@1={h:.0f}, EM={em:.0f}')

## 11. Extract GNN Features cho Phase 4

Day la buoc **quan trong** de proposed method co the dung GNN nhu heuristic.

Can extract tu trained GNN model:
1. **Node scores** per question (pred_dist)
2. **Entity embeddings** (local_entity_emb sau GNN)
3. **Top candidates** voi scores

In [ ]:
# === Extract GNN features cho CA HAI splits ===

def extract_gnn_features(info_path, data_path):
    """Extract GNN features tu .info file + .json data.
    Dung cho bat ky split nao (train/test).
    """
    features = {}
    
    data_list = []
    with open(data_path) as f:
        for line in f:
            data_list.append(json.loads(line))
    
    gnn_outputs = []
    with open(info_path) as f:
        for line in f:
            gnn_outputs.append(json.loads(line))
    
    for data, gnn_out in zip(data_list, gnn_outputs):
        qid = data['id']
        candidates = gnn_out.get('cand', [])
        
        node_scores = {}
        for entity, score in candidates:
            node_scores[entity] = score
        
        features[qid] = {
            'question': data['question'],
            'node_scores': node_scores,
            'top_candidates': [c[0] for c in candidates[:20]],
            'q_entities': data.get('entities', []),
            'answers': data.get('answers', []),
            'subgraph_tuples': data['subgraph']['tuples'],
            'gnn_f1': gnn_out.get('f1', 0),
            'gnn_hit': gnn_out.get('hit', 0),
        }
    
    return features


# === Extract TEST features ===
TEST_INFO = os.path.join(GNN_RAG_DIR, 'llm', 'results', 'gnn',
                          'RoG-webqsp', 'rearev-sbert', 'test.info')
TEST_DATA = os.path.join(GNN_RAG_DIR, 'gnn', 'data', 'webqsp', 'test.json')

if os.path.exists(TEST_INFO) and os.path.exists(TEST_DATA):
    test_features = extract_gnn_features(TEST_INFO, TEST_DATA)
    save_path = os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(test_features, f)
    print(f'[TEST] Saved {len(test_features)} questions to {save_path}')
    print(f'  Avg F1: {np.mean([v["gnn_f1"] for v in test_features.values()]):.4f}')
else:
    print('Test files not available.')

# === Extract TRAIN features (QUAN TRONG cho Phase 4) ===
TRAIN_INFO = os.path.join(GNN_RAG_DIR, 'gnn', 'checkpoint', 'pretrain', 'train.info')
TRAIN_DATA = os.path.join(GNN_RAG_DIR, 'gnn', 'data', 'webqsp', 'train.json')

if os.path.exists(TRAIN_INFO) and os.path.exists(TRAIN_DATA):
    train_features = extract_gnn_features(TRAIN_INFO, TRAIN_DATA)
    save_path = os.path.join(CHECKPOINT_DIR, 'gnn_features_train.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(train_features, f)
    print(f'[TRAIN] Saved {len(train_features)} questions to {save_path}')
    print(f'  Avg F1: {np.mean([v["gnn_f1"] for v in train_features.values()]):.4f}')
else:
    print(f'Train files not available.')
    print(f'  train.info: {os.path.exists(TRAIN_INFO)} <- Chay cell "6b" truoc!')
    print(f'  train.json: {os.path.exists(TRAIN_DATA)}')

## 12. Critical Analysis: Diem manh & yeu cua GNN-RAG

### Diem manh:
1. **Multi-hop reasoning**: ReaRev voi 9 hops (3 iter x 3 gnn) reach xa trong KG
2. **Question-conditioned**: Instructions encode question -> GNN aggregate theo question context
3. **Instruction update**: Query representation thay doi -> adaptive focus
4. **Local subgraph**: Moi question co graph rieng -> khong bi noise tu global KG

### Diem yeu (motivation cho proposed method):

| Limitation | Chi tiet | RL co the giai quyet? |
|-----------|---------|----------------------|
| **One-shot scoring** | GNN score 1 lan -> lay top -> done. Khong adapt sau khi thay paths | Co: RL re-evaluate sau moi buoc |
| **Shortest path cung** | Luon lay shortest path, khong xet content tren path | Co: RL chon path theo content |
| **Khong co STOP** | Luon lay tat ca paths co the, khong biet khi nao du | Co: RL hoc STOP action |
| **Union gay noise** | RA union nhieu nguon paths -> prompt dai, gay nhieu | Co: RL + greedy chon paths |
| **Khong co budget** | Khong gioi han so paths/tokens | Co: Budget constraint trong RL |
| **Supervised only** | Train voi answer labels, khong toi uu retrieval truc tiep | Co: RL toi uu end-to-end |

### Key insight cho Phase 4:
- GNN scores la **excellent heuristic** (Hits@1 ~76%)
- Nhung scoring la **one-shot**, khong **sequential**
- => Dung GNN scores lam **features** cho RL policy
- => RL quyet dinh: di dau tiep, khi nao dung, chon paths nao

In [ ]:
# === Phan tich cu the tu data ===

if os.path.exists(TEST_INFO):
    # Load tat ca GNN outputs
    all_gnn = []
    with open(TEST_INFO) as f:
        for line in f:
            all_gnn.append(json.loads(line))
    
    f1s = [g.get('f1', 0) for g in all_gnn]
    hits = [g.get('hit', 0) for g in all_gnn]
    n_cands = [len(g.get('cand', [])) for g in all_gnn]
    
    print('=== GNN-RAG Statistics ===')
    print(f'Total questions: {len(all_gnn)}')
    print(f'Avg F1: {np.mean(f1s):.4f}')
    print(f'Avg Hits@1: {np.mean(hits):.4f}')
    print(f'Avg candidates: {np.mean(n_cands):.1f}')
    print(f'F1=0 (total fail): {sum(1 for f in f1s if f == 0)} / {len(f1s)}')
    print(f'Hits=0 (top1 wrong): {sum(1 for h in hits if h == 0)} / {len(hits)}')
    
    # Failure cases
    print(f'\n=== Failure Analysis ===')
    fail_no_cand = sum(1 for g in all_gnn if len(g.get('cand', [])) == 0)
    fail_wrong_top = sum(1 for g in all_gnn if g.get('hit', 0) == 0 and len(g.get('cand', [])) > 0)
    print(f'No candidates: {fail_no_cand}')
    print(f'Wrong top-1 (but has candidates): {fail_wrong_top}')
else:
    print('test.info not available')

## Checklist truoc khi sang Phase 3

- [ ] Da hieu ReaRev architecture (multi-iter, multi-gnn, instruction update)
- [ ] Da hieu data format (local subgraph per question, JSON lines)
- [ ] Da hieu pipeline: GNN score -> shortest paths -> verbalize -> LLM
- [ ] Da hieu entity-level evaluation (F1, Hits@1, eps threshold)
- [ ] Da train/load pre-trained GNN model
- [ ] Da export GNN features (gnn_features_webqsp.pkl) cho Phase 4
- [ ] Da phan tich diem manh/yeu -> clear motivation cho RL